In [1]:
import pandas as pd
ratings = pd.read_csv(
    "./ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"]
)
ratings = ratings[["userId", "movieId", "timestamp", "rating"]]
ratings.head()

,userId,movieId,timestamp,rating
0,1,1193,978300760,5
1,1,661,978302109,3
2,1,914,978301968,3
3,1,3408,978300275,4
4,1,2355,978824291,5


In [2]:
user_ids = ratings["userId"].unique()
movie_ids = ratings["movieId"].unique()

user2idx = {u:i for i, u in enumerate(user_ids)}
movie2idx = {m:i for i, m in enumerate(movie_ids)}

ratings["user"] = ratings["userId"].map(user2idx)
ratings["movie"] = ratings["movieId"].map(movie2idx)

ratings["rating"] = ratings["rating"].astype("float32")
ratings["rating"] = (ratings["rating"] - ratings["rating"].mean()) / ratings["rating"].std()

In [3]:
ratings

,userId,movieId,timestamp,rating,user,movie
0,1,1193,978300760,1.269746,0,0
1,1,661,978302109,-0.520601,0,1
2,1,914,978301968,-0.520601,0,2
3,1,3408,978300275,0.374572,0,3
4,1,2355,978824291,1.269746,0,4
...,...,...,...,...,...,...
1000204,6040,1091,956716541,-2.310948,6039,772
1000205,6040,1094,956704887,1.269746,6039,1106
1000206,6040,562,956704746,1.269746,6039,365
1000207,6040,1096,956715648,0.374572,6039,152


In [4]:
num_users = ratings["user"].nunique()
num_movies = ratings["movie"].nunique()

print(num_users, num_movies)

6040 3706


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [6]:
class RatingDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df["user"].values, dtype=torch.long)
        self.movies = torch.tensor(df["movie"].values, dtype=torch.long)
        self.ratings = torch.tensor(df["rating"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]

In [7]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=42)

train_dataset = RatingDataset(train_df)
test_dataset = RatingDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

In [24]:
class NCF(nn.Module):
    def __init__(self, num_users, num_movies, emb_size=100, hidden_sizes=[64,32,16]):
        super().__init__()
                    
        self.user_emb = nn.Embedding(num_users, emb_size)
        self.movie_emb = nn.Embedding(num_movies, emb_size)
        
                    
        layers = []
        input_size = emb_size*2
        for h in hidden_sizes:
            layers.append(nn.Linear(input_size, h))
            layers.append(nn.ReLU())
            input_size = h
        self.mlp = nn.Sequential(*layers)
        
                      
        self.output = nn.Linear(input_size, 1)
        
    def forward(self, user, movie):
        u = self.user_emb(user)
        m = self.movie_emb(movie)
        x = torch.cat([u, m], dim=1)
        x = self.mlp(x)
        x = self.output(x).squeeze()
        return x

In [26]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = NCF(num_users, num_movies, emb_size=50).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
loss_fn = nn.MSELoss()

epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for user, movie, rating in train_loader:
        user = user.to(device)
        movie = movie.to(device)
        rating = rating.to(device)

        pred = model(user, movie)
        loss = loss_fn(pred, rating)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(user)
    
    print(f"Epoch {epoch+1} | Train Loss: {total_loss/len(train_dataset):.4f}")

Epoch 1 | Train Loss: 0.8357
Epoch 2 | Train Loss: 0.6890
Epoch 3 | Train Loss: 0.6578
Epoch 4 | Train Loss: 0.6444
Epoch 5 | Train Loss: 0.6349
Epoch 6 | Train Loss: 0.6266
Epoch 7 | Train Loss: 0.6195
Epoch 8 | Train Loss: 0.6119
Epoch 9 | Train Loss: 0.6035
Epoch 10 | Train Loss: 0.5940
Epoch 11 | Train Loss: 0.5833
Epoch 12 | Train Loss: 0.5712
Epoch 13 | Train Loss: 0.5585
Epoch 14 | Train Loss: 0.5455
Epoch 15 | Train Loss: 0.5323
Epoch 16 | Train Loss: 0.5193
Epoch 17 | Train Loss: 0.5064
Epoch 18 | Train Loss: 0.4943
Epoch 19 | Train Loss: 0.4825
Epoch 20 | Train Loss: 0.4712


In [27]:
import math
model.eval()
with torch.no_grad():
    all_preds = []
    all_ratings = []
    for user, movie, rating in test_loader:
        user = user.to(device)
        movie = movie.to(device)
        rating = rating.to(device)

        pred = model(user, movie)
        all_preds.append(pred.cpu())
        all_ratings.append(rating.cpu())
        
    all_preds = torch.cat(all_preds)
    all_ratings = torch.cat(all_ratings)
    mse = F.mse_loss(all_preds, all_ratings)
    rmse = math.sqrt(mse.item())
    print("Test RMSE:", rmse)

Test RMSE: 0.8467372359943227


In [28]:
            
user_tensor = torch.tensor([1]*num_movies).to(device)
movie_tensor = torch.arange(num_movies).to(device)

with torch.no_grad():
    scores = model(user_tensor, movie_tensor)

top10 = torch.topk(scores, 10).indices.cpu().numpy()

In [29]:
movies = pd.read_csv(
    "ml-1m/movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["movieId", "title", "genres"],
    header=None
)
                     
movie2idx = {m:i for i, m in enumerate(movies["movieId"].unique())}
idx2movie = {i:m for m,i in movie2idx.items()}

                         
idx2title = {i: movies[movies["movieId"]==mid]["title"].values[0] for i, mid in idx2movie.items()}

In [30]:
import numpy as np

                     
loo_train_movies = {}                                
loo_test_movie = {}                        

for user_id in ratings["user"].unique():
    user_movies = ratings[ratings["user"]==user_id].sort_values("timestamp")
    loo_test_movie[user_id] = int(user_movies.iloc[-1]["movie"])
    loo_train_movies[user_id] = user_movies.iloc[:-1]["movie"].values

In [31]:
user_id = 1
liked_movie_idx = loo_test_movie[user_id]
liked_movie_title = idx2title[liked_movie_idx]

print("User 1 left-out test movie:")
print("-", liked_movie_title)

User 1 left-out test movie:
- Down Periscope (1996)


In [35]:
model.eval()
with torch.no_grad():
    user_tensor = torch.tensor([user_id]*num_movies).to(device)
    movie_tensor = torch.arange(num_movies).to(device)
    scores = model(user_tensor, movie_tensor)

                                
train_movies_user1 = loo_train_movies[user_id]
scores[train_movies_user1] = -1e9

                               
top10 = torch.topk(scores, 30).indices.cpu().numpy()
top10_titles = [idx2title[i] for i in top10]

print("\nTop 10 recommended movies for user 1:")
for rank, title in enumerate(top10_titles, 1):
    print(f"{rank}. {title} → predicted score: {scores[top10[rank-1]].item():.3f}")


Top 10 recommended movies for user 1:
1. Child's Play (1988) → predicted score: 1.161
2. This Is Spinal Tap (1984) → predicted score: 1.114
3. Better Off Dead... (1985) → predicted score: 1.090
4. Stop! Or My Mom Will Shoot (1992) → predicted score: 1.080
5. M (1931) → predicted score: 1.044
6. Desperately Seeking Susan (1985) → predicted score: 1.026
7. Judgment Night (1993) → predicted score: 1.025
8. Bonheur, Le (1965) → predicted score: 1.006
9. War Stories (1995) → predicted score: 0.995
10. Red Violin, The (Le Violon rouge) (1998) → predicted score: 0.987
11. Rage: Carrie 2, The (1999) → predicted score: 0.985
12. Angel on My Shoulder (1946) → predicted score: 0.965
13. Billy's Holiday (1995) → predicted score: 0.958
14. Lovers of the Arctic Circle, The (Los Amantes del Círculo Polar) (1998) → predicted score: 0.956
15. Yojimbo (1961) → predicted score: 0.956
16. Losing Chase (1996) → predicted score: 0.952
17. Finding North (1999) → predicted score: 0.950
18. Maximum Overdrive 

In [34]:
K = 30
recalls = []

for user_id in ratings["user"].unique():
    test_movie = loo_test_movie[user_id]
    train_movies = loo_train_movies[user_id]

    user_tensor = torch.tensor([user_id]*num_movies).to(device)
    movie_tensor = torch.arange(num_movies).to(device)

    with torch.no_grad():
        scores = model(user_tensor, movie_tensor)
    
    scores[train_movies] = -1e9
    topk = torch.topk(scores, K).indices.cpu().numpy()

    hit = 1 if test_movie in topk else 0
    recalls.append(hit)

avg_recall_at_k = np.mean(recalls)
print("\nAverage Recall@k (Leave-One-Out):", avg_recall_at_k)


Average Recall@k (Leave-One-Out): 0.033443708609271525
